In [1]:
# %%
import requests
from bs4 import BeautifulSoup
import time
import csv

## 2. Set Up the Scraper

We define the base URL of the website and headers to mimic a browser request.  
This helps in preventing blocking from the website.

In [2]:
# Function to scrape data from one page
BASE_URL = "http://quotes.toscrape.com"

headers = {"User-Agent": "Mozilla/5.0"}


## 3. Create Helper Function

This function sends a request to the given URL and returns a BeautifulSoup object.  
It also handles errors in case the request fails.

In [3]:
# %%
def get_soup(url):
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except Exception as e:
        print(f"Error: {e}")
        return None

## 4. Scrape Single Job Listing

This function extracts details of a single job listing such as:
- Job Title  
- Company Name  
- Location

In [4]:
# %%
def scrape_job(item):
    try:
        job_title = item.find("span", class_="text").text
        company = item.find("small", class_="author").text
        
        # Dummy location logic
        location = "India"

        return {
            "title": job_title,
            "company": company,
            "location": location
        }
    except:
        return None

## 5. Get Job Listings from a Page

This function extracts all job listings from a single webpage.  
It identifies the relevant HTML elements and returns them for further processing.

In [5]:
# %%
def get_job_items(page_url):
    soup = get_soup(page_url)
    if not soup:
        return []

    return soup.find_all("div", class_="quote")

## 6. Find Total Number of Pages

This function determines how many pages are available for scraping.  
For simplicity, a fixed number of pages is used.

In [6]:
# %%
def get_total_pages():
    return 5

## 7. Scrape All Job Data

This function loops through all pages and extracts job data using previously defined functions.  

- Stores all job listings in a list  
- Uses delay between requests  
- Stops when maximum items are reached (if specified)

In [7]:
# %%
def scrape_all_jobs(max_items=None):
    all_jobs = []
    count = 0

    total_pages = get_total_pages()

    for page in range(1, total_pages + 1):
        print(f"Scraping page {page}...")

        url = f"{BASE_URL}/page/{page}/"
        items = get_job_items(url)

        for item in items:
            job = scrape_job(item)
            if job:
                all_jobs.append(job)
                count += 1

                if max_items and count >= max_items:
                    return all_jobs

        time.sleep(1)

    return all_jobs

## 8. Display Job Listings

In this step, we display the scraped job listings in a structured format.  
This helps in verifying that the scraping process is working correctly.

In [8]:
# %%
jobs = scrape_all_jobs(20)

print("\n" + "="*60)
print("JOB LISTINGS")
print("="*60)

for i, job in enumerate(jobs, 1):
    print(f"\n{i}. {job['title']}")
    print(f"   Company: {job['company']}")
    print(f"   Location: {job['location']}")

Scraping page 1...
Scraping page 2...

JOB LISTINGS

1. “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
   Company: Albert Einstein
   Location: India

2. “It is our choices, Harry, that show what we truly are, far more than our abilities.”
   Company: J.K. Rowling
   Location: India

3. “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
   Company: Albert Einstein
   Location: India

4. “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
   Company: Jane Austen
   Location: India

5. “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
   Company: Marilyn Monroe
   Location: India

6. “Try not to become a man of success. Rather become a man of value.”
   Company: Albert Einstein
   Location: India

7. “It is better to be hated f

## 9. Save Data to CSV File

This function saves the scraped job data into a CSV file.

- Creates a file named job_listings.csv  
- Writes column headers  
- Stores job details in rows

In [9]:
# %%
def save_jobs_to_csv(jobs, filename="job_listings.csv"):
    if not jobs:
        print("No data to save")
        return

    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow(["Job Title", "Company", "Location"])

        for job in jobs:
            writer.writerow([
                job["title"],
                job["company"],
                job["location"]
            ])

    print(f"Data saved to {filename}")

## 10. Execute Scraping and Save Data

In this step, we:
1. Call the scraping function to collect job data  
2. Check the number of jobs scraped  
3. Save the data into a CSV file  


In [10]:
# %%
save_jobs_to_csv(jobs)


Data saved to job_listings.csv


## 11. Read CSV File

This step reads the saved CSV file and displays its contents.  
It helps verify that the data has been stored correctly.

In [11]:
# %%
with open("job_listings.csv", "r", encoding="utf-8") as file:
    print(file.read())

Job Title,Company,Location
“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”,Albert Einstein,India
"“It is our choices, Harry, that show what we truly are, far more than our abilities.”",J.K. Rowling,India
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”,Albert Einstein,India
"“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”",Jane Austen,India
"“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”",Marilyn Monroe,India
“Try not to become a man of success. Rather become a man of value.”,Albert Einstein,India
“It is better to be hated for what you are than to be loved for what you are not.”,André Gide,India
"“I have not failed. I've just found 10,000 ways that won't work.”",Thomas A. Edison,India
“A woman is like a tea bag; you n